In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"
  from IPython.display import set_matplotlib_formats
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L3NoYXJlZC90ZWFtL2luZmx1ZW56YS10ZXN0aW5nL25mbG93LXRlc3QvYmlu'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/importlib/_bootstrap.py": 1750064009.0605173, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/importlib/_bootstrap_external.py": 1750064009.0585172, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/zipimport.py": 1750064008.3005228, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/codecs.py": 1750064007.9875252, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/encodings/aliases.py": 1750064008.538521, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/encodings/__init__.py": 1750064008.6585202, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/encodings/utf_8.py": 1750064008.7855194, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flusub/lib/python3.13/abc.py": 1750064007.9585254, "/shared/team/conda/katehowell.gpha-ukhsa-mscap/mscape-flus

In [2]:
climb_id= 'TEST_ID'
datapath= './flusub.mapped.tab'

In [3]:
# Injected Parameters
climb_id = "C-A8D4B7D902"
datapath = "/shared/team/influenza-testing/nflow-test/mapping-test/C-A8D4B7D902/flusub.mapped.tab"


In [4]:
from collections import Counter
import pandas as pd
import os,sys,glob
cols = ['query_name','query_len','query_start_coord','query_end_coord','strand','target_name','target_length','target_start','target_end','num_base_mapped','num_bases_gaps_mapped','mapping_quality','x1','x2','x3','x4','x5','x6']

#for the different segments

samp = ""

segments = ['PB2','PB1','PA','HA','MP','NA','NP','NS']
estlens = [2341,2341,2233,1778, 1565, 1413,1027,890]

mapresults =pd.read_csv(datapath,sep="\t",header=None)

readsdict_seg = {}
segmentdict = {}

    
def split_read_hit_info(mapresults, test):
    mapresults['target_subtype']=test[2]
    mapresults['target_segment'] = test[0]
    mapresults['target_isolate'] = test[1]

    test = mapresults["target_subtype"].str.split(pat="N", expand=True)
    mapresults['hit_Htype']=test[0]
    mapresults['hit_Ntype']=test[1]
    mapresults['hit_Htype'] = mapresults['hit_Htype'].str.replace("H","")
    mapresults['hit_Htype'] = mapresults['hit_Htype'].str.replace("A/","")
hacoverage = 0
nacoverage = 0
if mapresults.shape[0]>=1:
    mapresults.columns = cols

    test = mapresults["target_name"].str.split(pat="|", expand=True)

    split_read_hit_info(mapresults, test)

    mapresults = mapresults.sort_values(by=['query_name','mapping_quality','num_base_mapped'], ascending=True)
    newdf = mapresults.drop_duplicates(subset = ['query_name'],keep = 'last').reset_index(drop = True)
    segsummary = []
    for n,s in enumerate(segments):
        seginfo = []
        segresults = newdf[newdf['target_segment']==s]
        seginfo.append(segresults.shape[0])
        Countervar = Counter(segresults['target_subtype'])
        if segresults.shape[0]>=1:
            subtypes = list(set(segresults['target_subtype']))
            covlist = []
            for st in subtypes:
                subsegresults = segresults[segresults['target_subtype']==st]
                covlist.append(f"{st}:{round(float(subsegresults['num_base_mapped'].sum())/estlens[n],2)}")
            if s == "HA":
                hacoverage = round(float(segresults['num_base_mapped'].sum())/estlens[n],2)
            if s == "NA":
                nacoverage = round(float(segresults['num_base_mapped'].sum())/estlens[n],2)
            seginfo.append(";".join(covlist))
        else:
            seginfo.append(0)
        if len(Countervar) >= 1:
            mutdf = pd.DataFrame.from_dict(Countervar, orient="index").reset_index()
            mutdf = mutdf.rename(columns={"index": "type", 0: "count"})
            typecountsamp = []
            types = []
            for index,row in mutdf.iterrows():
                typecountsamp.append(f"{row['type']}:{row['count']}")
            segsummary.append("; ".join(list(set(typecountsamp))))
            seginfo.append("; ".join(list(set(typecountsamp))))
        else:
            segsummary.append(0)
            seginfo.append(0)

        segmentdict[s] = seginfo
    readsdict_seg[samp] = segsummary


segreadstab = pd.DataFrame.from_dict(segmentdict, orient='index')
segreadstab.columns = ['Num_reads','Estimated_coverage','Subtype_mapped']

segreadstab['Segment'] = segreadstab.index
segreadstab.to_csv(f"{climb_id}_segment_mapping_summary.csv")

if hacoverage>=0.9:
    clade_call = "Clade call possible"
else:
    clade_call = "Coverage too low for clade call"

ha = segreadstab[segreadstab['Segment']=='HA']
na = segreadstab[segreadstab['Segment']=='NA']
mixed_results_HA = False
mixed_results_NA = False

ha_seasonal = ['1','3',1,3]
ha_seasonal_flag = False
if ha.Num_reads[0] ==0:
    HAtype = 'H-'
else:
    hanewdf = newdf[newdf['target_segment']=='HA']
    subtypes = list(set(hanewdf['hit_Htype']))

    if len(subtypes)>1:
        #choose subtype
        multsubtypes = ";".join(subtypes)
        HAtype=f'H{multsubtypes}'
        mixed_results_HA = True
    else:
        HAtype=f'H{subtypes[0]}'
        if subtypes[0] in ha_seasonal:
            ha_seasonal_flag = True
if ha.Num_reads[0] <=2:
    HAconfidence = 'Low'
elif ha.Num_reads[0] <=5:
    HAconfidence = 'Medium'
else:
    HAconfidence = 'High'
na_seasonal = ['1','2',1,2]
na_seasonal_flag = False
NAtype = '-'
if na.Num_reads[0] ==0:
    NAtype = 'N-'
else:
    nanewdf = newdf[newdf['target_segment']=='NA']
    nasubtypes = list(set(nanewdf['hit_Ntype']))
    if len(nasubtypes)>1:
       #choose subtype
        mixed_results_NA = True
        nasubtypes = ";".join(nasubtypes)
    else:
        NAtype=f'N{nasubtypes[0]}'
        if nasubtypes[0] in na_seasonal:
            na_seasonal_flag = True
if na.Num_reads[0] <=2:
    NAconfidence = 'Low'
elif na.Num_reads[0] <=5:
    NAconfidence = 'Medium'
else:
    NAconfidence = 'High'
subtype_call = f'{HAtype}{NAtype}'
seasonal = False
if ha_seasonal_flag:
    if na_seasonal_flag:
        seasonal=True
hacall = ''
nacall = ''

if mixed_results_HA:
    hacall = 'Mixed subtypes detected'
elif ha_seasonal_flag:
    hacall = 'Seasonal'
elif HAtype == 'H-':
    hacall = 'No reads detected'
else:
    hacall = 'Non-seasonal influenza HA reads detected'

if mixed_results_NA:
    nacall = 'Mixed subtypes detected'
elif na_seasonal_flag:
    nacall = 'Seasonal NA reads'
elif NAtype == 'N-':
    nacall = 'No reads detected'
else:
    nacall = 'Non-seasonal influenza NA reads detected'

if HAconfidence == 'Low':
    res_conf = 'Low'
if NAconfidence == 'Low':
    res_conf = 'Low'

if HAconfidence == 'Medium':
    if NAconfidence in ['Medium','High']:
        res_conf = 'Medium'

if HAconfidence == 'High':
    if NAconfidence == 'High':
        res_conf = 'High'

/tmp/ipykernel_41203/816023821.py:94: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if ha.Num_reads[0] ==0:
/tmp/ipykernel_41203/816023821.py:109: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if ha.Num_reads[0] <=2:
/tmp/ipykernel_41203/816023821.py:118: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if na.Num_reads[0] ==0:
/tmp/ipykernel_41203/816023821.py:131: FutureWarning: Series.__getitem__ treating keys as positions is depreca

In [5]:
def generate_status_json():
    return_dict = {
        "analysis_id": None,
        "published_date": datetime.datetime.now().strftime("%Y-%m-%d"),
        "analysis_date": datetime.datetime.now().strftime("%Y-%m-%d"),
        "name": "flu_subtyping",
        "details": "Flu subtyping – minimap2 and clade typing",
        "exit_code": 0,
        "pipeline_status": "success",
        "result": {
            "flu_subtype": subtype_call,
            "fluHA_category": hacall,
            "fluNA_category": nacall,
            "HA_reads": str(ha.Num_reads[0]),
            "NA_reads": str(na.Num_reads[0]),
            "HA_coverage": str(hacoverage),
            "NA_coverage": str(nacoverage),
            "flu_clade": clade_call,
            "rag_status": res_conf
        },
        "report": "s3://",
        "outputs": "s3://",
        "identifiers": [],
        "records": [climb_id]
    }

    return return_dict
import json
import datetime

# Serializing json
json_object = json.dumps(generate_status_json())

# Writing to sample.json
with open(f"{climb_id}_flusub_result.json", "w") as outfile:
    outfile.write(json_object)

/tmp/ipykernel_41203/542254362.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "HA_reads": str(ha.Num_reads[0]),
/tmp/ipykernel_41203/542254362.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  "NA_reads": str(na.Num_reads[0]),


In [6]:
#| label: tab-segments

segreadstab

,Num_reads,Estimated_coverage,Subtype_mapped,Segment
PB2,1,A/H1N1pdm09:0.36,A/H1N1pdm09:1,PB2
PB1,3,A/H1N1pdm09:0.76;A/H1N1:0.19,A/H1N1:1; A/H1N1pdm09:2,PB1
PA,0,0,0,PA
HA,0,0,0,HA
MP,0,0,0,MP
NA,0,0,0,NA
NP,0,0,0,NP
NS,0,0,0,NS
